In [1]:
%pip install xgboost hyperopt


In [2]:
from hyperopt import hp
import pandas as pd
from hyperopt import STATUS_OK
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from hyperopt import fmin, tpe, Trials

In [3]:
taxi = pd.read_csv('taxi.csv', index_col=0)

In [4]:
taxi_pred = pd.read_csv('taxi_pred.csv', index_col = 0)

In [ ]:
taxi['pickup_wday'] = taxi['pickup_wday'].replace({
    1: 'Mon',
    2: 'Tue',
    3: 'Wed',
    4: 'Thu',
    5: 'Fri',
    6: 'Sat',
    7: 'Sun'
})

In [6]:
taxi_pred['pickup_wday'] = taxi_pred['pickup_wday'].replace({
    1: 'Mon',
    2: 'Tue',
    3: 'Wed',
    4: 'Thu',
    5: 'Fri',
    6: 'Sat',
    7: 'Sun'
})

In [7]:
from pandas.tseries.holiday import USFederalHolidayCalendar

airport_zones = ['JFK Airport', 'LaGuardia Airport']
calendar = USFederalHolidayCalendar()

def is_airport_ride(pickup, dropoff):
    return pickup in airport_zones or dropoff in airport_zones

def is_holiday(date):
    holidays = calendar.holidays()
    return pd.to_datetime(date).normalize() in holidays

def is_weekend(day_name):
    weekend = ['Sat', 'Sun']
    return day_name in weekend


taxi['is_airport_ride'] = taxi.apply(lambda row: is_airport_ride(row['pickup_zone'], row['dropoff_zone']), axis=1)
taxi['is_holiday'] = taxi['pickup_date'].apply(is_holiday)
taxi['is_weekend'] = taxi['pickup_wday'].apply(is_weekend)

taxi_pred['is_airport_ride'] = taxi_pred.apply(lambda row: is_airport_ride(row['pickup_zone'], row['dropoff_zone']), axis=1)
taxi_pred['is_holiday'] = taxi_pred['pickup_date'].apply(is_holiday)
taxi_pred['is_weekend'] = taxi_pred['pickup_wday'].apply(is_weekend)

In [8]:
taxi.drop(['manhattan_distance', 'distance_diff', 'pair', 'pickup_BoroCode', 'dropoff_BoroCode', 'pickup_NTACode', 'dropoff_NTACode', 'cost_per_mile' , 'cost_per_minute', 'pickup_month', 'vendor_id', 'pickup_week', 'pickup_doy', 'distance_bin', 'pickup_wday'], axis=1, inplace=True)

In [9]:
taxi.drop(['pickup_Borough', 'dropoff_Borough', 'pickup_zone',	'dropoff_zone'], axis = 1, inplace = True)

In [10]:
taxi_pred.drop(['manhattan_distance', 'distance_diff', 'pair', 'pickup_BoroCode', 'dropoff_BoroCode', 'pickup_NTACode', 'dropoff_NTACode', 'pickup_month' , 'vendor_id', 'pickup_week',	'pickup_doy', 'distance_bin', 'pickup_wday'], axis=1, inplace=True)

In [11]:
taxi_pred.drop(['pickup_Borough', 'dropoff_Borough', 'pickup_zone',	'dropoff_zone'], axis = 1, inplace = True)

In [12]:
taxi.head()

,pickup_date,response,pickup_hour,length_time,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,fare_amount,speed,time_of_day,is_airport_ride,is_holiday,is_weekend
0,2015-05-11,1.75,18,520,-73.983009,40.766602,-73.967972,40.760677,1,0.90,7.0,6.23,Evening,False,False,False
1,2015-05-06,2.95,18,824,-73.989510,40.734470,-73.997185,40.717976,1,1.40,10.0,6.12,Evening,False,False,False
2,2015-05-11,2.46,7,1016,-74.012741,40.702229,-73.998878,40.713711,1,1.14,11.5,4.04,Morning,False,False,False
3,2015-05-13,1.66,11,550,-73.970909,40.767231,-73.979507,40.752331,1,1.28,7.5,8.38,Morning,False,False,False
4,2015-05-07,1.75,20,501,-73.973869,40.747925,-73.986618,40.730328,1,1.40,7.5,10.06,Evening,False,False,False


In [13]:
taxi_pred.head()

,pickup_date,pickup_hour,length_time,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,fare_amount,speed,time_of_day,is_airport_ride,is_holiday,is_weekend
0,2015-05-24,7,807,-73.951279,40.769939,-73.973137,40.759491,1,2.27,10.5,10.13,Morning,False,False,True
1,2015-05-21,0,1458,-73.998993,40.730064,-73.959808,40.710918,1,3.20,18.0,7.90,Night,False,False,False
2,2015-05-04,23,495,-73.994148,40.717796,-73.948204,40.722595,1,3.40,12.0,24.73,Night,False,False,False
3,2015-05-13,19,941,-73.979813,40.743240,-73.952881,40.780380,2,2.93,13.0,11.21,Evening,False,False,False
4,2015-05-01,20,141,-73.983543,40.726109,-73.988991,40.729980,1,0.47,3.5,12.00,Evening,False,False,False


In [14]:
categorical_features = [col for col in taxi.columns if taxi[col].dtype == 'object']
categorical_features_pred = [col for col in taxi_pred.columns if taxi_pred[col].dtype == 'object']

In [15]:
taxi = pd.get_dummies(taxi, columns=categorical_features)
taxi_pred = pd.get_dummies(taxi_pred, columns=categorical_features_pred)

In [16]:
X = taxi.drop('response', axis=1)
y = taxi['response']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, shuffle = True, random_state=42)

In [17]:
space = {
    'max_depth': hp.choice('max_depth', range(1, 10)),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.choice('n_estimators', range(10, 300)),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
    'subsample': hp.uniform('subsample', 0.5, 1),
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 1, 5),
}


In [18]:
def objective(params):
    clf = XGBRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        gamma=params['gamma'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        objective='reg:absoluteerror'
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    return {'loss': mae, 'status': STATUS_OK}

In [20]:
trials = Trials()
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=100,
            trials=trials)

print("Best: ", best)


100%|██████████| 100/100 [23:38<00:00, 14.19s/trial, best loss: 0.560493963756125]
Best:  {'colsample_bytree': 0.9990240162746576, 'gamma': 4.796714813478375, 'learning_rate': 0.16560939853275353, 'max_depth': 6, 'n_estimators': 121, 'reg_alpha': 0.19484949987464328, 'reg_lambda': 4.253351365860161, 'subsample': 0.9881778964662438}


In [21]:
best_model = XGBRegressor(**best, objective='reg:absoluteerror', random_state=42)
best_model.fit(X_train, y_train)
final_preds = best_model.predict(X_test)
final_mae = mean_absolute_error(y_test, final_preds)

print(f'Final MAE on Test Set: {final_mae:.4f}')

Final MAE on Test Set: 0.5610


In [22]:
best_model.fit(X, y)
predictions = best_model.predict(taxi_pred)

In [23]:
export_pred = pd.DataFrame(predictions, columns = ['response'])

export_pred.to_csv('HyperOpt_XGBoost_FLAGS_AE.txt', index = False, header = False)